# Yahoo Pipeline Run and Debug Notebook

Use this notebook for two tasks:

1. Run Yahoo pipeline stages in pieces.
2. Debug failures quickly when a stage fails.

Run order for piecewise flow: Cells 2 to 8.

## 1) Setup

In [29]:
import json
import os
from pathlib import Path

import polars as pl
from pyiceberg.catalog import load_catalog

from nfl.yahoo_fantasy.api import YahooApiClient, iter_dicts
from nfl.yahoo_fantasy.auth import build_oauth_session, load_token
from nfl.yahoo_fantasy.pipeline import PipelineConfig, run_pipeline, _resolve_weeks
from nfl.yahoo_fantasy.transforms import transform
from nfl.yahoo_fantasy.storage.iceberg import IcebergCatalogConfig

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

## 2) Inputs and Run Controls

In [30]:
# Required inputs
LEAGUE_KEY = "461.l.717896"
SPORT = "nfl"

# Run controls
TARGET_SEASON = 2025
START_WEEK = 1
END_WEEK = 17
USE_CACHE = False
INCLUDE_UNROSTERED_PLAYER_STATS = True

print("Inputs ready")
print(f"  league_key={LEAGUE_KEY}")
print(f"  sport={SPORT}")
print(f"  season={TARGET_SEASON}, weeks={START_WEEK}-{END_WEEK}")

Inputs ready
  league_key=461.l.717896
  sport=nfl
  season=2025, weeks=1-17


## 3) Environment Bootstrap

In [31]:
# Project bootstrap and catalog initialization
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise RuntimeError("Cannot locate project root (missing pyproject.toml).")

project_root = find_project_root(Path.cwd())
os.chdir(project_root)

CATALOG_URI = "sqlite:///iceberg_catalog.db"
WAREHOUSE = "./warehouse"
catalog = load_catalog("yahoo", type="sql", uri=CATALOG_URI, warehouse=WAREHOUSE)

print("Environment ready")
print(f"  cwd={Path.cwd()}")
print(f"  catalog_uri={CATALOG_URI}")
print(f"  warehouse={WAREHOUSE}")

Environment ready
  cwd=C:\Users\EricTruett\nfl
  catalog_uri=sqlite:///iceberg_catalog.db
  warehouse=./warehouse


## 4) Auth and API Client

In [32]:
# OAuth session setup
token_path = project_root / ".secrets" / "yahoo_token.json"
credentials_path = project_root / ".secrets" / "credentials.json"
cached_token = load_token(token_path)

file_creds = {}
if credentials_path.exists():
    with open(credentials_path, "r", encoding="utf-8") as f:
        file_creds = json.load(f)

client_id = os.getenv("YAHOO_CLIENT_ID") or str(file_creds.get("client_id", "")).strip()
client_secret = os.getenv("YAHOO_CLIENT_SECRET") or str(file_creds.get("client_secret", "")).strip()
redirect_uri = os.getenv("YAHOO_REDIRECT_URI") or str(file_creds.get("redirect_uri", "")).strip() or "http://localhost:8000"

if not client_id or not client_secret or client_id.startswith("YOUR_") or client_secret.startswith("YOUR_"):
    raise ValueError("Set valid Yahoo OAuth credentials in env vars or .secrets/credentials.json")

oauth_session = build_oauth_session(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri=redirect_uri,
    token_path=token_path,
    auth_code=os.getenv("YAHOO_AUTH_CODE", ""),
    open_browser=False,
 )

client = YahooApiClient(
    oauth_session=oauth_session,
    use_cache=USE_CACHE,
    validate_contracts=True,
 )

print("OAuth and API client ready")
print(f"  use_cache={USE_CACHE}")
print(f"  token_cached={cached_token is not None}")

OAuth and API client ready
  use_cache=False
  token_cached=True


## 5) Piece 1: Fetch Common Entities

In [33]:
# Piece 1: common entities
league = client.get_league_metadata(LEAGUE_KEY)
if TARGET_SEASON and int(league.get("season") or 0) != TARGET_SEASON:
    print(f"Warning: league season is {league.get('season')} but TARGET_SEASON={TARGET_SEASON}")

common_entities = {
    "league": [league],
    "team": client.get_teams(LEAGUE_KEY),
    "player": client.get_players(LEAGUE_KEY),
    "draft_pick": client.get_draft_picks(LEAGUE_KEY, season=int(league["season"])),
    "transaction": client.get_transactions(LEAGUE_KEY, season=int(league["season"])),
    "stat_category": client.get_stat_categories(LEAGUE_KEY, game_id=int(league["game_id"])),
    "scoring_rule": client.get_scoring_rules(LEAGUE_KEY, season=int(league["season"])),
}

print("Common entities fetched")
for k, v in common_entities.items():
    print(f"  {k}: {len(v)}")

Common entities fetched
  league: 1
  team: 12
  player: 1187
  draft_pick: 168
  transaction: 624
  stat_category: 86
  scoring_rule: 30


## 6) Piece 2: Fetch NFL Entities

In [34]:
# Piece 2: NFL entities
weeks = list(range(START_WEEK, END_WEEK + 1))
team_keys = sorted({str(t.get("team_key") or "") for t in common_entities["team"] if t.get("team_key")})

nfl_entities = {
    "standings": client.get_standings(LEAGUE_KEY, sport=SPORT),
    "matchups": client.get_matchups(LEAGUE_KEY, season=int(league["season"]), weeks=weeks),
    "roster_entries": client.get_roster_entries(
        LEAGUE_KEY,
        season=int(league["season"]),
        weeks=weeks,
        team_keys=team_keys,
    ),
}

base_weekly = client.get_player_stats_weekly(
    LEAGUE_KEY,
    season=int(league["season"]),
    roster_entries=nfl_entities["roster_entries"],
)

if INCLUDE_UNROSTERED_PLAYER_STATS:
    all_weekly = client.get_player_stats_weekly_all_players(
        LEAGUE_KEY,
        season=int(league["season"]),
        weeks=weeks,
    )
    rows_by_key = {(r["league_key"], r["week"], r["player_key"]): dict(r) for r in base_weekly}
    for r in all_weekly:
        rows_by_key[(r["league_key"], r["week"], r["player_key"])] = dict(r)
    nfl_entities["player_stats_weekly"] = sorted(rows_by_key.values(), key=lambda r: (r["week"], r["player_key"]))
else:
    nfl_entities["player_stats_weekly"] = base_weekly

print("NFL entities fetched")
for k, v in nfl_entities.items():
    print(f"  {k}: {len(v)}")

NFL entities fetched
  standings: 12
  matchups: 196
  roster_entries: 2968
  player_stats_weekly: 19414


## 7) Piece 3: Transform and Validate

In [35]:
# Piece 3: Transform and validate scoring health
frames = transform(common_entities=common_entities, nfl_entities=nfl_entities, nba_entities={})

weekly_stats = frames["nfl_player_stats_weekly"]
weekly_health = (
    weekly_stats
    .group_by(["season", "week"])
.agg([
        pl.len().alias("player_rows"),
        (pl.col("fantasy_points").fill_null(0) != 0).sum().alias("non_zero_player_rows"),
    ])
    .with_columns(
        pl.when(pl.col("player_rows") > 0)
        .then((pl.col("non_zero_player_rows") * 100.0) / pl.col("player_rows"))
        .otherwise(0.0)
        .round(2)
        .alias("pct_non_zero")
    )
    .sort(["week"])
)

print("Transform complete")
print(f"  nfl_player_stats_weekly rows={weekly_stats.height}")
print(weekly_health)

Transform complete
  nfl_player_stats_weekly rows=19414
shape: (17, 5)
┌────────┬──────┬─────────────┬──────────────────────┬──────────────┐
│ season ┆ week ┆ player_rows ┆ non_zero_player_rows ┆ pct_non_zero │
│ ---    ┆ ---  ┆ ---         ┆ ---                  ┆ ---          │
│ i64    ┆ i64  ┆ u32         ┆ u32                  ┆ f64          │
╞════════╪══════╪═════════════╪══════════════════════╪══════════════╡
│ 2025   ┆ 1    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 2    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 3    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 4    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 5    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 6    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 7    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 8    ┆ 1187        ┆ 601                  ┆ 50.63        │
│ 2025   ┆ 9    ┆ 1

## 8) Optional Full Pipeline Run

In [ ]:
# Optional: full pipeline run (all-in-one)
full_cfg = PipelineConfig(
    storage_target="none",
    use_cache=USE_CACHE,
    validate_contracts=True,
    require_nfl_player_points=False,
    include_nfl_unrostered_player_stats=INCLUDE_UNROSTERED_PLAYER_STATS,
    start_week=START_WEEK,
    end_week=END_WEEK,
)

try:
    full_result = run_pipeline(
        league_key=LEAGUE_KEY,
        sport=SPORT,
        oauth_session=oauth_session,
        config=full_cfg,
    )
    print("Full pipeline run completed")
    print(f"  Frames: {list(full_result.frames.keys())}")
except Exception as exc:
    print("Full pipeline run failed")
    print(type(exc).__name__, str(exc))

No cache directory found.
Pipeline configuration ready.
  league_key: 461.l.717896
  sport: nfl
  storage_target: none
  include_nfl_unrostered_player_stats: True
  start_week: 1
  end_week: 17

Yahoo request denied/rate-limited on uncached fetch. Retrying with cache enabled...


RuntimeError: Yahoo request denied for '/league/461.l.717896/settings' (status=999). Retry later or reduce request rate.

## 9) Team and Player Scoring Output

In [ ]:
# Team + player scoring view from transformed piecewise frames
if "frames" not in globals():
    raise RuntimeError("Run Cell 7 first to build transformed frames")

stats_df = frames["nfl_player_stats_weekly"].unique(subset=["league_key", "season", "week", "player_key"], keep="first")
roster_df = frames["nfl_roster_entries"].unique(subset=["league_key", "season", "week", "player_key"], keep="first")
team_df = frames["team"]
player_df = frames["player"]

team_player_scoring = (
    stats_df
    .join(
        roster_df.select(["league_key", "season", "week", "player_key", "team_key", "selected_position"]),
        on=["league_key", "season", "week", "player_key"],
        how="left",
    )
    .join(team_df.select(["league_key", "team_key", "team_name", "owner_name"]), on=["league_key", "team_key"], how="left")
    .join(player_df.select(["player_key", "full_name", "display_position"]), on="player_key", how="left")
    .with_columns([
        pl.coalesce([pl.col("team_name"), pl.lit("Unrostered / Unknown")]).alias("team_name"),
        pl.coalesce([pl.col("owner_name"), pl.lit("N/A")]).alias("owner_name"),
        pl.coalesce([pl.col("full_name"), pl.col("player_key")]).alias("player_name"),
        pl.coalesce([pl.col("selected_position"), pl.col("display_position"), pl.lit("N/A")]).alias("position"),
    ])
    .select(["season", "week", "team_name", "owner_name", "player_name", "position", "fantasy_points", "status"])
)

print(team_player_scoring.filter(pl.col("season") == TARGET_SEASON).sort(["week", "team_name", "fantasy_points"], descending=[False, False, True]).head(50))

Weekly player scoring health for 2025 (weeks 1-17):
shape: (17, 5)
┌────────┬──────┬─────────────┬──────────────────────┬──────────────┐
│ season ┆ week ┆ player_rows ┆ non_zero_player_rows ┆ pct_non_zero │
│ ---    ┆ ---  ┆ ---         ┆ ---                  ┆ ---          │
│ i64    ┆ i64  ┆ u32         ┆ u32                  ┆ f64          │
╞════════╪══════╪═════════════╪══════════════════════╪══════════════╡
│ 2025   ┆ 1    ┆ 346         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 2    ┆ 344         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 3    ┆ 348         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 4    ┆ 342         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 5    ┆ 346         ┆ 0                    ┆ 0.0          │
│ …      ┆ …    ┆ …           ┆ …                    ┆ …            │
│ 2025   ┆ 13   ┆ 354         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 14   ┆ 352         ┆ 0                    ┆ 0.0          │
│ 2025   ┆ 15   ┆ 348  

## 10) Advanced Debug Probe (Run Only If Needed)

In [ ]:
# Advanced debug: probe Yahoo team roster scoring payloads
probe_client = YahooApiClient(
    oauth_session=oauth_session,
    use_cache=False,
    validate_contracts=False,
)

probe_league = probe_client.get_league_metadata(LEAGUE_KEY)
probe_teams = probe_client.get_teams(LEAGUE_KEY)

if not probe_teams:
    print("No teams found for probe.")
else:
    probe_team_key = str(probe_teams[0].get("team_key") or "")
    candidate_weeks = list(range(START_WEEK, END_WEEK + 1))
    probe_payload = None
    probe_week = None
    probe_path = None
    last_error = None

    for week in candidate_weeks:
        for path in [
            f"/team/{probe_team_key}/roster;week={week}/players/stats",
            f"/team/{probe_team_key}/roster;week={week};out=players,stats",
            f"/team/{probe_team_key}/roster;week={week};out=players",
        ]:
            try:
                probe_payload = probe_client.get(path, use_cache=False)
                probe_week = week
                probe_path = path
                break
            except Exception as exc:
                last_error = exc
        if probe_payload is not None:
            break

    if probe_payload is None:
        print("Probe failed")
        print(f"  Last error: {last_error}")
    else:
        player_points_nodes = 0
        player_stats_nodes = 0
        for node in iter_dicts(probe_payload):
            if isinstance(node, dict):
                if "player_points" in node:
                    player_points_nodes += 1
                if "player_stats" in node:
                    player_stats_nodes += 1

        print(f"Probe endpoint: {probe_path}")
        print(f"Probe week: {probe_week}")
        print(f"player_points nodes: {player_points_nodes}")
        print(f"player_stats nodes: {player_stats_nodes}")